Cellule 1 : Configuration et Fonctions Utiles

In [ ]:
import mercury as mr
import pandas as pd
import openpyxl
import re
import io
import os
import time

# Configuration de l'App Mercury
app = mr.App(title="🚀 Harness Data Engine", 
            description="Analyse intelligente de fichiers usines",
            show_code=False)

def get_param(param_str, key):
    try:
        parts = [p.strip() for p in str(param_str).split(',')]
        for p in parts:
            if p.startswith(key):
                return p.split(':', 1)[1].strip().replace('"', '').replace('[', '').replace(']', '')
        return None
    except: return None
      
def match_col(pattern, col_list):
    p_clean = pattern.replace('*', '').lower().strip()
    for col in col_list:
        col_str = str(col).lower().strip()
        if pattern.endswith('*') and col_str.startswith(p_clean): return col
        elif col_str == p_clean: return col
    return None

Cellule 2 : Chargement des fichiers

In [ ]:
# Chargement du fichier de règles (local)
rules_path = "Rules.xlsx"
if not os.path.exists(rules_path):
    mr.Markdown("❌ **Erreur : Rules.xlsx manquant.**")
    mr.Stop()

rules_df = pd.read_excel(rules_path).sort_values('Ordre')

# Widget d'upload Mercury
data_file = mr.File(label="📂 Déposez votre fichier Excel ici")

if data_file.filepath is None:
    # Message d'accueil dynamique basé sur le premier MESSAGE de l'Excel
    welcome_row = rules_df[rules_df['Action'] == 'MESSAGE'].head(1)
    if not welcome_row.empty:
        mr.Markdown(f"### {welcome_row['Message / Question'].values[0]}")
    mr.Stop()

Cellule 3 : Le Moteur Complet (Le Pipeline)

In [ ]:
# Initialisation de l'état (Variables globales au Notebook)
if 'variables_sac' not in locals(): variables_sac = {}
if 'df_principal' not in locals(): df_principal = None

mr.Markdown("---")
mr.Markdown("## 💬 Journal de traitement")

for idx, row in rules_df.iterrows():
    action = row['Action']
    cible = row['Cible']
    params = str(row['Paramètres'])
    on_fail = row['Si Échec']
    msg_excel = str(row['Message / Question'])

    try:
        # --- MESSAGE ---
        if action == "MESSAGE":
            txt = msg_excel
            for var, val in variables_sac.items():
                txt = txt.replace(f"{{{var}}}", str(val))
            mr.Markdown(f"**Assistant :** {txt}")

        # --- FIND_METADATA ---
        elif action == "FIND_METADATA":
            wb = openpyxl.load_workbook(data_file.filepath, data_only=True)
            ws = wb.active
            keyword = get_param(params, "keyword")
            found = False
            for r in range(1, 25):
                for c in range(1, 12):
                    val = ws.cell(r, c).value
                    if val and keyword in str(val):
                        variables_sac[cible] = ws.cell(r, c+1).value
                        found = True; break
                if found: break
            if found:
                mr.Markdown(f"ℹ️ Métadonnée : **{cible}** détectée ({variables_sac[cible]})")
            else: raise Exception(f"Clé '{keyword}' introuvable")

        # --- LOAD_DATA_TABLE ---
        elif action == "LOAD_DATA_TABLE":
            if df_principal is not None: continue
            
            start_key = get_param(params, "Start_At")
            m_param = get_param(params, "Mandatory")
            o_param = get_param(params, "Optional")
            mandatory_raw = [c.strip() for c in m_param.split(';')] if m_param else []
            optional_raw = [c.strip() for c in o_param.split(';')] if o_param else []
            
            temp = pd.read_excel(data_file.filepath, header=None, nrows=50)
            mask = temp.apply(lambda r: r.astype(str).str.contains(start_key, case=False, na=False).any(), axis=1)
            if not mask.any(): raise Exception(f"Clé de départ '{start_key}' introuvable.")
            
            header_idx = temp[mask].index[0]
            actual_columns = pd.read_excel(data_file.filepath, skiprows=header_idx, nrows=0).columns.tolist()
            
            columns_to_keep = []
            for m in mandatory_raw:
                if not m: continue
                found = match_col(m, actual_columns)
                if found and found not in columns_to_keep: columns_to_keep.append(found)
                elif not found: raise Exception(f"❌ Colonne obligatoire '{m}' introuvable.")
            for o in optional_raw:
                if not o: continue
                found = match_col(o, actual_columns)
                if found and found not in columns_to_keep: columns_to_keep.append(found)

            df_principal = pd.read_excel(data_file.filepath, skiprows=header_idx, usecols=columns_to_keep)
            mr.Markdown(f"📊 Tableau chargé : {len(df_principal)} lignes.")

        # --- RENAME ---
        elif action == "RENAME":
            new_name = get_param(params, "To")
            if new_name in df_principal.columns: continue
            real = match_col(cible, df_principal.columns)
            if real:
                df_principal.rename(columns={real: new_name}, inplace=True)
                mr.Markdown(f"🏷️ Colonne renommée : `{real}` ➔ `{new_name}`")
            elif on_fail == "STOP": raise Exception(f"Colonne '{cible}' introuvable.")

        # --- INJECT ---
        elif action == "INJECT":
            var_key = get_param(params, "Value")
            val = variables_sac.get(var_key, "N/A")
            df_principal[cible] = val
            mr.Markdown(f"💉 Injection : Colonne `{cible}` créée avec `{val}`")

        # --- FILTER ---
        elif action == "FILTER":
            cond_raw = get_param(params, "Condition").strip().strip('"').strip("'")
            found_op = None
            for op in ['!=', '==', '>=', '<=', '>', '<']:
                if op in cond_raw:
                    found_op = op
                    break
            if not found_op: raise Exception(f"Opérateur manquant : {cond_raw}")
            
            parts = cond_raw.split(found_op)
            col_left = match_col(parts[0].strip().replace('`', ''), df_principal.columns)
            col_right = match_col(parts[1].strip().replace('`', ''), df_principal.columns)
            
            nb_avant = len(df_principal)
            if found_op == '!=':
                df_principal = df_principal[df_principal[col_left].astype(str).str.strip() != df_principal[col_right].astype(str).str.strip()]
            elif found_op == '==':
                df_principal = df_principal[df_principal[col_left].astype(str).str.strip() == df_principal[col_right].astype(str).str.strip()]
            mr.Markdown(f"✂️ Filtrage : {nb_avant} ➔ {len(df_principal)} lignes.")

        # --- CONTROLE_CORRECT ---
        elif action == "CONTROLE_CORRECT":
            fmt = get_param(params, "Format")
            real_col = match_col(cible, df_principal.columns)
            n = int(fmt.split(':')[1]) if "chars:" in fmt else 3
            mask_invalid = df_principal[real_col].astype(str).str.strip().str.len() != n
            
            if mask_invalid.any():
                invalid_df = df_principal.loc[mask_invalid, real_col].unique()
                mr.Markdown(f"⚠️ **Corrections requises pour {cible}**")
                for val in invalid_df:
                    # Ici, on utilise le widget Mercury pour la saisie
                    fix = mr.Text(label=f"Correction pour '{val}'", value="")
                    if fix.value == "":
                        mr.Markdown(f"⏸️ En attente de saisie pour `{val}`")
                        mr.Stop()
                    if fix.value.upper() == "DELETE":
                        df_principal = df_principal[df_principal[real_col] != val]
                    else:
                        df_principal.loc[df_principal[real_col] == val, real_col] = fix.value

        # --- CONTROLE ---
        elif action == "CONTROLE":
            fmt = get_param(params, "Format")
            val_raw = variables_sac.get(cible, None)
            if val_raw is None and df_principal is not None:
                real_col = match_col(cible, df_principal.columns)
                if real_col: val_raw = df_principal[real_col].iloc[0]
            
            val_to_check = str(val_raw).strip()
            is_valid, reason = False, ""
            
            if "exact_digits:" in fmt:
                n = int(fmt.split(':')[1]); is_valid = val_to_check.isdigit() and len(val_to_check) == n; reason = f"{n} digits"
            elif "alphanum_fixed:" in fmt:
                n = int(fmt.split(':')[1]); is_valid = bool(re.match(rf"^[A-Za-z0-9]{{{n}}}$", val_to_check)); reason = f"{n} alphanum"
            elif fmt == "alphanum_code":
                is_valid = bool(re.match(r"^[A-Za-z0-9\s]+$", val_to_check)); reason = "alphanumerical"
            elif "max_chars:" in fmt:
                n = int(fmt.split(':')[1]); is_valid = len(" ".join(val_to_check.split())) <= n; reason = f"max {n} chars"
            elif "chars:" in fmt:
                n = int(fmt.split(':')[1]); is_valid = len(val_to_check) == n; reason = f"{n} chars"
            elif fmt == "numeric_price":
                is_valid = bool(re.match(r"^\d+(\.\d{1,3})?$", val_to_check.replace(',', '.'))); reason = "price"

            icon = "✅" if is_valid else "❌"
            mr.Markdown(f"🔍 **{cible}** : {val_to_check} {icon} ({reason})")
            if not is_valid and on_fail == "STOP": raise Exception(f"Échec {cible}")

    except Exception as e:
        mr.Markdown(f"⚠️ **Erreur ({action})** : {e}")
        if on_fail == "STOP": mr.Stop()

Cellule 4 : Résultat et Export

In [ ]:
if df_principal is not None:
    mr.Markdown("### ✅ Analyse terminée !")
    mr.PandasTable(df_principal)
    
    output = io.BytesIO()
    with pd.ExcelWriter(output, engine='openpyxl') as writer:
        df_principal.to_excel(writer, index=False)
    
    # Mercury permet l'export direct via le bouton de l'interface ou lien
    with open("Resultat.xlsx", "wb") as f:
        f.write(output.getvalue())
    mr.Markdown("📥 [Cliquez ici pour télécharger le Resultat.xlsx](./Resultat.xlsx)")